In [1]:
import nltk
from nltk.corpus import treebank
import re
import math

# -----------------------------
#  DOWNLOAD DATA
# -----------------------------
nltk.download('treebank')

# -----------------------------
#  LOAD CORPUS
# -----------------------------
tagged_sents = treebank.tagged_sents()
tagged_words = treebank.tagged_words()

# -----------------------------
# CORPUS DETAILS
# -----------------------------
print("\n===== CORPUS DETAILS =====")
print("Total Sentences:", len(tagged_sents))
print("Total Words:", len(tagged_words))

print("\nSample Sentence:")
print(tagged_sents[0])

# -----------------------------
# TAG LIST (COLUMN FORMAT)
# -----------------------------
print("\n===== TAG LIST (COLUMN FORMAT) =====")

tag_list = sorted(set(tag for word, tag in tagged_words))
print("Total Tags:", len(tag_list))

cols = 5
rows = math.ceil(len(tag_list) / cols)

for i in range(rows):
    row_items = []
    for j in range(cols):
        index = i + j * rows
        if index < len(tag_list):
            row_items.append(f"{tag_list[index]:<6}")
    print(" ".join(row_items))


# -----------------------------
# FULL TAG MEANINGS
# -----------------------------
print("\n===== COMPLETE TAG DESCRIPTIONS =====")

tag_meanings = {
    "CC": "Coordinating conjunction (and, but, or)",
    "CD": "Cardinal number (one, two, 100)",
    "DT": "Determiner (a, an, the)",
    "EX": "Existential 'there' (there is, there are)",
    "FW": "Foreign word",
    "IN": "Preposition or subordinating conjunction",
    "JJ": "Adjective (big, red)",
    "JJR": "Adjective, comparative (bigger)",
    "JJS": "Adjective, superlative (biggest)",
    "LS": "List item marker (1, A, a)",
    "MD": "Modal (can, could, will, would)",
    "NN": "Noun, singular (dog, car)",
    "NNS": "Noun, plural (dogs, cars)",
    "NNP": "Proper noun, singular (India, John)",
    "NNPS": "Proper noun, plural (Americans)",
    "PDT": "Predeterminer (all, both)",
    "POS": "Possessive ending ('s)",
    "PRP": "Personal pronoun (I, you, he)",
    "PRP$": "Possessive pronoun (my, your)",
    "RB": "Adverb (quickly, very)",
    "RBR": "Adverb, comparative (faster)",
    "RBS": "Adverb, superlative (fastest)",
    "RP": "Particle (up, off)",
    "SYM": "Symbol (+, %, &)",
    "TO": "to (infinitive marker)",
    "UH": "Interjection (oh, wow)",
    "VB": "Verb, base form (run)",
    "VBD": "Verb, past tense (ran)",
    "VBG": "Verb, gerund/present participle (running)",
    "VBN": "Verb, past participle (eaten)",
    "VBP": "Verb, non-3rd person singular present (run)",
    "VBZ": "Verb, 3rd person singular present (runs)",
    "WDT": "Wh-determiner (which, that)",
    "WP": "Wh-pronoun (who, what)",
    "WP$": "Possessive wh-pronoun (whose)",
    "WRB": "Wh-adverb (where, when)"
}

print(f"{'TAG':<6} {'MEANING'}")
print("-" * 60)

for tag in sorted(tag_meanings):
    print(f"{tag:<6} {tag_meanings[tag]}")

# -----------------------------
# RULE LIST
# -----------------------------
print("\n===== RULE LIST =====")

rules = [
    ("Rule 1", "Digits only", "CD"),
    ("Rule 2", "a, an, the", "DT"),
    ("Rule 3", "Prepositions", "IN"),
    ("Rule 4", "Pronouns", "PRP"),
    ("Rule 5", "Ends with 'ing'", "VBG"),
    ("Rule 6", "Ends with 'ed'", "VBD"),
    ("Rule 7", "Ends with 'ly'", "RB"),
    ("Rule 8", "Adj suffix", "JJ"),
    ("Rule 9", "Ends with 's'", "NNS"),
    ("Rule 10", "Capitalized", "NNP"),
    ("Rule 11", "From lexicon", "Learned"),
    ("Default", "Unknown word", "NN")
]

print(f"{'RULE':<10} {'CONDITION':<25} {'TAG':<10}")
print("-" * 50)

for rule in rules:
    print(f"{rule[0]:<10} {rule[1]:<25} {rule[2]:<10}")

# -----------------------------
# TRAIN-TEST SPLIT
# -----------------------------
split = int(0.8 * len(tagged_sents))
train_sents = tagged_sents[:split]
test_sents = tagged_sents[split:]

# -----------------------------
# BUILD LEXICON
# -----------------------------
word_tag_freq = {}

for sent in train_sents:
    for word, tag in sent:
        word = word.lower()
        if word not in word_tag_freq:
            word_tag_freq[word] = {}
        if tag not in word_tag_freq[word]:
            word_tag_freq[word][tag] = 0
        word_tag_freq[word][tag] += 1

lexicon = {}
for word in word_tag_freq:
    lexicon[word] = max(word_tag_freq[word], key=word_tag_freq[word].get)

# -----------------------------
# RULE-BASED TAGGER
# -----------------------------
def rule_based_tagger(sentence):
    tagged_sentence = []

    for word in sentence:
        w = word.lower()

        if re.match(r'^\d+$', w):
            tag = 'CD'
        elif w in ['a', 'an', 'the']:
            tag = 'DT'
        elif w in ['in', 'on', 'at', 'by', 'with', 'from', 'to', 'of']:
            tag = 'IN'
        elif w in ['i', 'you', 'he', 'she', 'it', 'we', 'they']:
            tag = 'PRP'
        elif w.endswith('ing'):
            tag = 'VBG'
        elif w.endswith('ed'):
            tag = 'VBD'
        elif w.endswith('ly'):
            tag = 'RB'
        elif w.endswith(('ous', 'ful', 'able', 'ive', 'al')):
            tag = 'JJ'
        elif w.endswith('s'):
            tag = 'NNS'
        elif word[0].isupper():
            tag = 'NNP'
        elif w in lexicon:
            tag = lexicon[w]
        else:
            tag = 'NN'

        tagged_sentence.append((word, tag))

    return tagged_sentence

# -----------------------------
# EVALUATION
# -----------------------------
def evaluate(test_data):
    total = 0
    correct = 0

    for sent in test_data:
        words = [word for word, tag in sent]
        true_tags = [tag for word, tag in sent]

        pred = rule_based_tagger(words)
        pred_tags = [tag for word, tag in pred]

        for t, p in zip(true_tags, pred_tags):
            total += 1
            if t == p:
                correct += 1

    return correct / total

# -----------------------------
# EVALUATE MODEL
# -----------------------------
print("\n===== EVALUATION =====")
accuracy = evaluate(test_sents)
print("Rule-Based Tagger Accuracy:", accuracy)

# -----------------------------
# TEST SENTENCE
# -----------------------------
print("\n===== TEST SENTENCE =====")
sentence = "Natural language processing is very interesting".split()

print("Sentence:", sentence)
print("Tagged:", rule_based_tagger(sentence))


[nltk_data] Downloading package treebank to /root/nltk_data...
[nltk_data]   Unzipping corpora/treebank.zip.



===== CORPUS DETAILS =====
Total Sentences: 3914
Total Words: 100676

Sample Sentence:
[('Pierre', 'NNP'), ('Vinken', 'NNP'), (',', ','), ('61', 'CD'), ('years', 'NNS'), ('old', 'JJ'), (',', ','), ('will', 'MD'), ('join', 'VB'), ('the', 'DT'), ('board', 'NN'), ('as', 'IN'), ('a', 'DT'), ('nonexecutive', 'JJ'), ('director', 'NN'), ('Nov.', 'NNP'), ('29', 'CD'), ('.', '.')]

===== TAG LIST (COLUMN FORMAT) =====
Total Tags: 46
#      CD     NN     RBS    VBZ   
$      DT     NNP    RP     WDT   
''     EX     NNPS   SYM    WP    
,      FW     NNS    TO     WP$   
-LRB-  IN     PDT    UH     WRB   
-NONE- JJ     POS    VB     ``    
-RRB-  JJR    PRP    VBD   
.      JJS    PRP$   VBG   
:      LS     RB     VBN   
CC     MD     RBR    VBP   

===== COMPLETE TAG DESCRIPTIONS =====
TAG    MEANING
------------------------------------------------------------
CC     Coordinating conjunction (and, but, or)
CD     Cardinal number (one, two, 100)
DT     Determiner (a, an, the)
EX     Existentia